In [4]:
import re                               # Regular expression: metin temizleme için
import pandas as pd                     # Veri çerçevesi (tabular veri) için
import numpy as np                      # Sayısal işlemler için
import matplotlib.pyplot as plt         # Grafik çizimi için
from transformers import AutoTokenizer  # Hugging Face tokenizer sınıfı
from datasets import Dataset            # Hugging Face veri seti sınıfı
import warnings                         # Uyarıları bastırmak için
warnings.filterwarnings('ignore')       # Gereksiz uyarıları kapat

print("Tüm kütüphaneler içe aktarıldı.")

Tüm kütüphaneler içe aktarıldı.


In [5]:
import pandas as pd

# 1. Veri setini yükleyelim
df = pd.read_csv('sentimentdataset.csv')
display(df.head(3))

# 2. Metin içeren sütunu tespit edelim (genelde Text, text vb. isimlendirilir)
text_col = next((col for col in df.columns if 'text' in col.lower() or 'tweet' in col.lower()), df.columns[0])
print(f"\nBulunan metin sütunu: '{text_col}'")

# 3. Metinleri listeye alalım. Çok uzun sürmemesi için örnek olarak ilk 6 tanesini 'raw_texts' değişkenine atıyoruz.
# İleride modeli eğitirken tamamını df[text_col].tolist() şeklinde alabilirsiniz.
raw_texts = df[text_col].dropna().astype(str).tolist()[:6]

# Her bir metni ekrana basalım, index numarasıyla
for i, text in enumerate(raw_texts):
    # display: zengin (rich) çıktı için; HTML ile renklendiriyoruz
    display(HTML(f"<b>[{i}]</b> <span style='color:gray;'>{text}</span>"))


,Unnamed: 0.1,Unnamed: 0,Text,Sentiment,Timestamp,User,Platform,Hashtags,Retweets,Likes,Country,Year,Month,Day,Hour
0,0,0,Enjoying a beautiful day at the park! ...,Positive,2023-01-15 12:30:00,User123,Twitter,#Nature #Park,15.0,30.0,USA,2023,1,15,12
1,1,1,Traffic was terrible this morning. ...,Negative,2023-01-15 08:45:00,CommuterX,Twitter,#Traffic #Morning,5.0,10.0,Canada,2023,1,15,8
2,2,2,Just finished an amazing workout! 💪 ...,Positive,2023-01-15 15:45:00,FitnessFan,Instagram,#Fitness #Workout,20.0,40.0,USA,2023,1,15,15



Bulunan metin sütunu: 'Text'


In [6]:
# ============================================================
# METİN TEMİZLEME FONKSİYONU
# ============================================================

# Her satırda ne yaptığımızı açıklayarak ilerliyoruz.

def clean_text(text: str) -> str:
    """
    Ham metni LLM girdisine hazır hale getirir.

    Parametre:
        text (str): Temizlenecek ham metin

    Dönüş:
        str: Temizlenmiş metin
    """

    # --- Adım 1: HTML etiketlerini temizle ---
    # re.sub(pattern, replacement, string): regex ile değiştirme yapar
    # r'<[^>]+>': '<' ile başlayan, '>' ile biten, içinde '>' olmayan her şeyi bulur
    text = re.sub(r'<[^>]+>', '', text)

    # --- Adım 2: URL'leri temizle ---
    # http:// veya https:// ile başlayan URL'leri bul ve sil
    text = re.sub(r'https?://\S+', '', text)

    # --- Adım 3: Emoji ve özel sembolleri temizle ---
    # Unicode kategorilerini kullanarak emoji/ sembol karakterlerini siliyoruz
    # \p{So}: Symbol Other (emoji gibi semboller)
    # \p{Cs}: Surrogate (emoji'lerin alt kısmı)
    # \p{Cn}: Unassigned (atanmamış karakterler)
    # re.UNICODE: Unicode desteğini açar
    # re.sub ile bu kategorilerdeki karakterleri '' (boş) yap
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"   # Emoticons (gülücükler)
        "\U0001F300-\U0001F5FF"   # Semboller ve piktogramlar
        "\U0001F680-\U0001F6FF"   # Ulaşım ve harita sembolleri
        "\U0001F1E0-\U0001F1FF"   # Bayraklar
        "\U00002702-\U000027B0"   # Dingbat'lar
        "\U000024C2-\U0001F251"   # Diğer semboller
        "]+", re.UNICODE)
    text = emoji_pattern.sub('', text)

    # --- Adım 4: Tekrarlayan noktalama işaretlerini teke indir ---
    # Örn: '!!!' -> '!', '???' -> '?', '...' -> '...' (3 nokta korunur)
    # ([!?.,]){2,}: Aynı noktalama işaretinin 2+ kez tekrarı
    # r'\1': sadece ilkini (1 tane) koy
    text = re.sub(r'([!?.,]){2,}', r'\1', text)

    # --- Adım 5: Fazla boşlukları temizle ---
    # \s+: bir veya daha fazla boşluk (space, tab, newline)
    # replace ile tek boşluk yap
    text = re.sub(r'\s+', ' ', text)

    # --- Adım 6: Baştaki ve sondaki boşlukları sil ---
    # strip() string'in başındaki ve sonundaki boşlukları temizler
    text = text.strip()

    return text


# Şimdi fonksiyonumuzu test edelim
print("=" * 70)
print("METİN TEMİZLEME SONUÇLARI")
print("=" * 70)

# Her bir ham metni temizleyip yan yana gösterelim
for i, text in enumerate(raw_texts):
    cleaned = clean_text(text)
    print(f"\n[{i}] ORİJİNAL:  {text}")
    print(f"    TEMİZ:     {cleaned}")

METİN TEMİZLEME SONUÇLARI

[0] ORİJİNAL:   Enjoying a beautiful day at the park!              
    TEMİZ:     Enjoying a beautiful day at the park!

[1] ORİJİNAL:   Traffic was terrible this morning.                 
    TEMİZ:     Traffic was terrible this morning.

[2] ORİJİNAL:   Just finished an amazing workout! 💪               
    TEMİZ:     Just finished an amazing workout!

[3] ORİJİNAL:   Excited about the upcoming weekend getaway!        
    TEMİZ:     Excited about the upcoming weekend getaway!

[4] ORİJİNAL:   Trying out a new recipe for dinner tonight.        
    TEMİZ:     Trying out a new recipe for dinner tonight.

[5] ORİJİNAL:   Feeling grateful for the little things in life.    
    TEMİZ:     Feeling grateful for the little things in life.


In [7]:
# ============================================================
# KENDİ BASİT BPE TOKENIZER'IMIZI YAZALIM
# ============================================================

# Bu bölümde BPE'nin mantığını anlamak için sıfırdan basit bir versiyon yazıyoruz.
# Gerçek projelerde Hugging Face tokenizer'ları kullanılır.

class SimpleBPE:
    """
    Basit bir Byte-Pair Encoding (BPE) tokenizer.
    Gerçek BPE'nin temel mantığını gösterir.
    """

    def __init__(self, vocab_size: int = 50):
        # vocab_size: hedef sözlük boyutu (kaç farklı token olacak)
        self.vocab_size = vocab_size
        # merges: hangi çiftlerin birleştiğini tutan liste
        self.merges = []
        # vocab: token -> id eşlemesi
        self.vocab = {}

    def get_stats(self, word: str) -> dict:
        # word içindeki bitişik karakter çiftlerinin frekansını hesaplar
        # Örn: "merhaba" -> {'me':1, 'er':1, 'rh':1, 'ha':1, 'ab':1, 'ba':1}
        stats = {}
        for i in range(len(word) - 1):
            pair = word[i] + word[i+1]
            stats[pair] = stats.get(pair, 0) + 1
        return stats

    def merge_pair(self, word: str, pair: str, new_token: str) -> str:
        # Belirli bir karakter çiftini (pair) yeni bir token (new_token) ile değiştir
        # Örn: word="merhaba", pair="me", new_token="X" -> "Xrhaba"
        result = ""
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] + word[i+1] == pair:
                # Eşleşen çifti bulduk: iki karakter yerine tek token yaz
                result += new_token
                i += 2  # iki karakter atla
            else:
                # Eşleşme yok: karakteri olduğu gibi ekle
                result += word[i]
                i += 1
        return result

    def fit(self, corpus: list):
        # corpus: eğitim metinleri listesi

        # Kelimeleri boşluktan ayır
        words = ' '.join(corpus).split()

        # Başlangıç: her karakter ayrı bir token
        # sorted(set(...)) benzersiz karakterleri alır
        chars = sorted(set(''.join(words)))
        # Her karaktere bir id ver (0, 1, 2, ...)
        self.vocab = {c: i for i, c in enumerate(chars)}

        # next_id: yeni birleştirilmiş token'lara vereceğimiz id
        next_id = len(self.vocab)

        # Tüm kelimeleri boşlukla ayırarak tek bir metin haline getir
        # BPE, kelime içi çiftleri birleştirir (kelime sınırlarını korur)
        text = ' '.join(words)

        print(f"Başlangıç vocab boyutu: {len(self.vocab)}")
        print(f"Hedef vocab boyutu: {self.vocab_size}")
        print(f"Eğitim metni ({len(text)} karakter): {text[:80]}...")
        print("\nBPE birleştirme adımları:")
        print("-" * 50)

        # Hedef vocab boyutuna ulaşana kadar en sık çifti birleştir
        step = 0
        while len(self.vocab) < self.vocab_size:
            # Mevcut metindeki tüm karakter çiftlerinin frekansını al
            stats = self.get_stats(text)

            # Hiç çift kalmadıysa dur
            if not stats:
                break

            # En sık geçen çifti bul (max fonksiyonu ile)
            # stats.items() -> (pair, freq) çiftlerini döndürür
            # key=lambda x: x[1] -> frekansa göre sırala
            best_pair = max(stats.items(), key=lambda x: x[1])[0]

            # Yeni token adı oluştur: merge edilen çift için sembol
            # chr(0xE000 + next_id) özel Unicode karakteri (Private Use Area)
            new_token = chr(0xE000 + next_id)

            # Birleştirme işlemini kaydet
            self.merges.append((best_pair, new_token))

            # Metindeki tüm best_pair örneklerini new_token ile değiştir
            text = self.merge_pair(text, best_pair, new_token)

            # Yeni token'ı vocab'a ekle
            self.vocab[new_token] = next_id
            next_id += 1

            step += 1
            # Her 5 adımda bir ilerlemeyi göster
            if step % 5 == 1 or step <= 3:
                print(f"  Adım {step}: '{best_pair}' -> '{new_token}' (frekans: {stats[best_pair]})")

        print("-" * 50)
        print(f"Son vocab boyutu: {len(self.vocab)}")
        print(f"Toplam birleştirme: {len(self.merges)}")

    def tokenize(self, text: str) -> list:
        # Verilen metni, öğrenilen merge kurallarına göre tokenize et
        # Önce kelimelere ayır
        words = text.split()
        tokens = []

        for word in words:
            # Her kelimeyi karakterlerine ayır
            current = ' '.join(list(word))
            # Her merge kuralını uygula
            for pair, new_token in self.merges:
                current = self.merge_pair(current, pair, new_token)
            # Boşlukla ayrılmış token'ları parçala
            tokens.extend(current.split())

        return tokens


# ============================================================
# BPE TOKENIZER'IMIZI TEST EDELİM
# ============================================================

corpus = [
    "merhaba dünya",
    "merhaba nasılsın",
    "dünya çok güzel",
    "nasılsın iyi misin",
    "güzel bir gün"
]

# vocab_size=35: 35 farklı token'a kadar birleştirme yap
bpe = SimpleBPE(vocab_size=35)
bpe.fit(corpus)

print("\n" + "=" * 50)
print("TEST TOKENIZASYONU")
print("=" * 50)

test_text = "merhaba dünya nasılsın"
print(f"Girdi: '{test_text}'")
print(f"Token'lar: {bpe.tokenize(test_text)}")

Başlangıç vocab boyutu: 19
Hedef vocab boyutu: 35
Eğitim metni (79 karakter): merhaba dünya merhaba nasılsın dünya çok güzel nasılsın iyi misin güzel bir gün...

BPE birleştirme adımları:
--------------------------------------------------
  Adım 1: 'a ' -> '' (frekans: 4)
  Adım 2: 'sı' -> '' (frekans: 4)
  Adım 3: 'ün' -> '' (frekans: 3)
  Adım 6: 'r' -> '' (frekans: 2)
  Adım 11: 'd' -> '' (frekans: 2)
  Adım 16: 'l' -> '' (frekans: 2)
--------------------------------------------------
Son vocab boyutu: 35
Toplam birleştirme: 16

TEST TOKENIZASYONU
Girdi: 'merhaba dünya nasılsın'
Token'lar: ['m', 'e', 'r', 'h', '\ue013b', 'a', 'd', 'ü', '\ue016y', 'a', '\ue016\ue013s', 'ı', 'l', 's', 'ı', 'n']


In [8]:
# ============================================================
# HUGGING FACE AUTO TOKENIZER YÜKLE
# ============================================================
# AutoTokenizer.from_pretrained: Önceden eğitilmiş tokenizer'ı indirir
# BERT tokenizer, WordPiece algoritması kullanır
# "bert-base-uncased": küçük harfe normalize eder, 30,522 vocab
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# BERT tokenizer'ında padding token zaten tanımlıdır ([PAD])
# GPT-2'den farklı olarak elle atama gerekmez
print(f"Model: BERT-base-uncased")
print(f"Vocab boyutu: {tokenizer.vocab_size:,}")
print(f"Maksimum uzunluk: {tokenizer.model_max_length:,}")
print(f"Pad token: '{tokenizer.pad_token}' (id: {tokenizer.pad_token_id})")
print(f"CLS token: '{tokenizer.cls_token}' (id: {tokenizer.cls_token_id})")
print(f"SEP token: '{tokenizer.sep_token}' (id: {tokenizer.sep_token_id})")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Model: BERT-base-uncased
Vocab boyutu: 30,522
Maksimum uzunluk: 512
Pad token: '[PAD]' (id: 0)
CLS token: '[CLS]' (id: 101)
SEP token: '[SEP]' (id: 102)


In [9]:
# ============================================================
# TOKENIZER'I TEK BİR METİN ÜZERİNDE TEST ET
# ============================================================
# Tokenizer'ın tüm özelliklerini görmek için örnek metni veri setinden alıyoruz
sample_text = df[text_col].dropna().astype(str).iloc[0]

# --- encode(): metni token id'lerine çevirir ---
# return_tensors="pt": PyTorch tensor'ı olarak döndür
encoded = tokenizer.encode(sample_text, return_tensors="pt")
print(f"Metin: {sample_text}")
print(f"\nToken ID'leri (PyTorch tensor): {encoded}")
print(f"Token ID shape: {encoded.shape}")  # [1, token_sayısı]

# --- decode(): token id'lerini metne çevirir ---
# .flatten(): tensor'ı düzleştir (1D dizi yap)
# .tolist(): Python listesine çevir
decoded = tokenizer.decode(encoded.flatten().tolist())
print(f"\nDecode edilmiş metin: {decoded}")

# --- tokenize(): metni token string'lerine ayırır (id'siz) ---
tokens = tokenizer.tokenize(sample_text)
print(f"\nTokenler ({len(tokens)} adet):")
print(tokens)

# Her token'in id'sini göster
print("\nToken -> ID eşlemesi:")
for token in tokens:
    token_id = tokenizer.convert_tokens_to_ids(token)
    print(f"  '{token}' -> {token_id}")


Metin:  Enjoying a beautiful day at the park!              

Token ID'leri (PyTorch tensor): tensor([[ 101, 9107, 1037, 3376, 2154, 2012, 1996, 2380,  999,  102]])
Token ID shape: torch.Size([1, 10])

Decode edilmiş metin: [CLS] enjoying a beautiful day at the park! [SEP]

Tokenler (8 adet):
['enjoying', 'a', 'beautiful', 'day', 'at', 'the', 'park', '!']

Token -> ID eşlemesi:
  'enjoying' -> 9107
  'a' -> 1037
  'beautiful' -> 3376
  'day' -> 2154
  'at' -> 2012
  'the' -> 1996
  'park' -> 2380
  '!' -> 999


In [10]:
# ============================================================
# PADDING, TRUNCATION VE ATTENTION MASK ÖRNEĞİ
# ============================================================

# Farklı uzunluklarda metinleri artık veri setimizden alıyoruz.
# Örneğin farklı uzunluklar görebilmek için 10. ile 13. satırlar arasını seçelim
texts = df[text_col].dropna().astype(str).tolist()[10:13]

print("Metinler ve karakter uzunlukları:")
for i, t in enumerate(texts):
    print(f"  [{i}] ({len(t):3d} kar) {t}")

# --- padding=True (en uzuna göre doldur) ---
# return_tensors="pt" ile PyTorch tensor'u olarak alıyoruz
encoded_pad = tokenizer(
    texts,                          # girdi metin listesi
    padding=True,                   # en uzun metne göre padding ekle
    truncation=False,               # truncation yapma
    return_tensors="pt"            # PyTorch tensor çıktısı
)

print("\n" + "=" * 70)
print("PADDING UYGULANMIŞ ÇIKTI")
print("=" * 70)
print(f"\ninput_ids shape: {encoded_pad['input_ids'].shape}")  # [3, max_len]
print(f"attention_mask shape: {encoded_pad['attention_mask'].shape}")

# Her bir metnin input_ids'ini göster
print("\ninput_ids:")
for i, ids in enumerate(encoded_pad['input_ids']):
    # ids bir tensor satırı; önce listeye çevir, sonra decode et
    decoded = tokenizer.decode(ids.tolist())
    print(f"  [{i}] {ids.tolist()}")
    print(f"       -> decode: '{decoded}'")

print("\nattention_mask:")
print("(1 = gerçek token, 0 = padding token)")
for i, mask in enumerate(encoded_pad['attention_mask']):
    gercek = mask.sum().item()  # 1'lerin sayısı = gerçek token sayısı
    pad = (mask == 0).sum().item()  # 0'ların sayısı = padding token sayısı
    print(f"  [{i}] {mask.tolist()} (gerçek: {gercek}, padding: {pad})")


Metinler ve karakter uzunlukları:
  [0] ( 52 kar)  Just published a new blog post. Check it out!      
  [1] ( 52 kar)  Feeling a bit under the weather today.             
  [2] ( 52 kar)  Exploring the city's hidden gems.                  

PADDING UYGULANMIŞ ÇIKTI

input_ids shape: torch.Size([3, 13])
attention_mask shape: torch.Size([3, 13])

input_ids:
  [0] [101, 2074, 2405, 1037, 2047, 9927, 2695, 1012, 4638, 2009, 2041, 999, 102]
       -> decode: '[CLS] just published a new blog post. check it out! [SEP]'
  [1] [101, 3110, 1037, 2978, 2104, 1996, 4633, 2651, 1012, 102, 0, 0, 0]
       -> decode: '[CLS] feeling a bit under the weather today. [SEP] [PAD] [PAD] [PAD]'
  [2] [101, 11131, 1996, 2103, 1005, 1055, 5023, 20296, 1012, 102, 0, 0, 0]
       -> decode: '[CLS] exploring the city ' s hidden gems. [SEP] [PAD] [PAD] [PAD]'

attention_mask:
(1 = gerçek token, 0 = padding token)
  [0] [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] (gerçek: 13, padding: 0)
  [1] [1, 1, 1, 1, 1, 1, 1, 1,

In [11]:
# ============================================================
# TRUNCATION ÖRNEĞİ
# ============================================================

# max_length ile maksimum token sayısını sınırlıyoruz
max_len = 8  # en fazla 8 token

encoded_trunc = tokenizer(
    texts,
    padding=True,               # kısaları doldur (8 tokene tamamla)
    truncation=True,            # uzunları kes
    max_length=max_len,         # maksimum token sayısı
    return_tensors="pt"
)

print(f"max_length = {max_len}")
print(f"input_ids shape: {encoded_trunc['input_ids'].shape}")  # [3, 8]
print("\nTruncation sonrası:")
for i, ids in enumerate(encoded_trunc['input_ids']):
    decoded = tokenizer.decode(ids.tolist())
    # decode edilince padding token'ları da görünebilir (<|endoftext|>)
    # attention_mask ile gerçek token'ları ayırt ederiz
    print(f"  [{i}] ID: {ids.tolist()}")
    print(f"       Metin: '{decoded}'")
    print(f"       Mask: {encoded_trunc['attention_mask'][i].tolist()}")
    print()

max_length = 8
input_ids shape: torch.Size([3, 8])

Truncation sonrası:
  [0] ID: [101, 2074, 2405, 1037, 2047, 9927, 2695, 102]
       Metin: '[CLS] just published a new blog post [SEP]'
       Mask: [1, 1, 1, 1, 1, 1, 1, 1]

  [1] ID: [101, 3110, 1037, 2978, 2104, 1996, 4633, 102]
       Metin: '[CLS] feeling a bit under the weather [SEP]'
       Mask: [1, 1, 1, 1, 1, 1, 1, 1]

  [2] ID: [101, 11131, 1996, 2103, 1005, 1055, 5023, 102]
       Metin: '[CLS] exploring the city ' s hidden [SEP]'
       Mask: [1, 1, 1, 1, 1, 1, 1, 1]

